# Notebook - create data

## libraries

In [ ]:
!pip instal l faker psycopg2-binary pandas

In [ ]:
from faker import Faker
import psycopg2
from psycopg2.extras import execute_batch
import random
import pandas as pd
from datetime import datetime

# Same data
Faker.seed(42)
random.seed(42)

## Database connection

In [ ]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="ecommerce",
    user="data_engineer",
    password="data_engineer_dev"
)

cursor = conn.cursor()

print("Successful connection!")

## Categories

In [ ]:
categories = [
    "Electronics",
    "Computers",
    "Smartphones",
    "Home Appliances",
    "Furniture",
    "Clothing",
    "Footwear",
    "Sports",
    "Beauty",
    "Books",
    "Gaming",
    "Accessories",
    "Office Supplies",
    "Toys",
    "Kitchen",
    "Garden",
    "Automotive",
    "Pet Supplies",
    "Health",
    "Music"
]

In [ ]:
category_ids = []

for category in categories:
    cursor.execute(
        """
        INSERT INTO categories (name)
        VALUES (%s)
        RETURNING category_id
        """,
        (category,)
    )

    category_id = cursor.fetchone()[0]
    category_ids.append(category_id)

conn.commit()

print(f"{len(category_ids)} categories inserted successfully!")

## Customers

In [ ]:
customer_ids = []

for _ in range(10000):

    first_name = fake.first_name()
    last_name = fake.last_name()
    email = fake.unique.email()

    cursor.execute(
        """
        INSERT INTO customers (
            first_name,
            last_name,
            email
        )
        VALUES (%s, %s, %s)
        RETURNING customer_id
        """,
        (
            first_name,
            last_name,
            email
        )
    )

    customer_id = cursor.fetchone()[0]
    customer_ids.append(customer_id)

conn.commit()

print(f"{len(customer_ids)} customers inserted successfully!")

## Products

In [ ]:
product_ids = []

for _ in range(1000):

    category_id = random.choice(category_ids)

    product_name = f"{fake.word().title()} {random.choice([
        'Laptop',
        'Smartphone',
        'Headphones',
        'Monitor',
        'Keyboard',
        'Mouse',
        'Camera',
        'Tablet',
        'Speaker',
        'Chair',
        'Backpack',
        'Watch',
        'Printer',
        'Desk'
    ])}"

    description = fake.paragraph(nb_sentences=3)

    price = round(random.uniform(10, 5000), 2)

    stock_quantity = random.randint(0, 500)

    cursor.execute(
        """
        INSERT INTO products (
            name,
            description,
            price,
            stock_quantity,
            category_id
        )
        VALUES (%s, %s, %s, %s, %s)
        RETURNING product_id
        """,
        (
            product_name,
            description,
            price,
            stock_quantity,
            category_id
        )
    )

    product_id = cursor.fetchone()[0]
    product_ids.append(product_id)

conn.commit()

print(f"{len(product_ids)} products inserted successfully!")

## Orders

In [ ]:
order_ids = []

order_statuses = [
    "pending",
    "paid",
    "processing",
    "shipped",
    "delivered",
    "cancelled"
]

for _ in range(30000):

    customer_id = random.choice(customer_ids)

    order_date = fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )

    status = random.choice(order_statuses)

    cursor.execute(
        """
        INSERT INTO orders (
            customer_id,
            order_date,
            status,
            total_amount
        )
        VALUES (%s, %s, %s, %s)
        RETURNING order_id
        """,
        (
            customer_id,
            order_date,
            status,
            0
        )
    )

    order_id = cursor.fetchone()[0]
    order_ids.append(order_id)

conn.commit()

print(f"{len(order_ids)} orders inserted successfully!")

## Order items

In [ ]:
order_ids = []

order_statuses = [
    "pending",
    "paid",
    "processing",
    "shipped",
    "delivered",
    "cancelled"
]

for _ in range(30000):

    customer_id = random.choice(customer_ids)

    order_date = fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )

    status = random.choice(order_statuses)

    cursor.execute(
        """
        INSERT INTO orders (
            customer_id,
            order_date,
            status,
            total_amount
        )
        VALUES (%s, %s, %s, %s)
        RETURNING order_id
        """,
        (
            customer_id,
            order_date,
            status,
            0
        )
    )

    order_id = cursor.fetchone()[0]
    order_ids.append(order_id)

conn.commit()

print(f"{len(order_ids)} orders inserted successfully!")

## Payments

In [ ]:
payment_methods = [
    "Credit Card",
    "Debit Card",
    "PayPal",
    "Bank Transfer",
    "Digital Wallet"
]

payment_statuses = [
    "Pending",
    "Approved",
    "Failed",
    "Refunded"
]

In [ ]:
for order_id in order_ids:

    cursor.execute(
        """
        SELECT total_amount
        FROM orders
        WHERE order_id = %s
        """,
        (order_id,)
    )

    amount = cursor.fetchone()[0]

    cursor.execute(
        """
        INSERT INTO payments (
            order_id,
            payment_method,
            payment_status,
            amount
        )
        VALUES (%s, %s, %s, %s)
        """,
        (
            order_id,
            random.choice(payment_methods),
            random.choice(payment_statuses),
            amount
        )
    )

conn.commit()

print("Payments inserted successfully!")

## Inventory

In [ ]:
for product_id in product_ids:

    quantity = random.randint(0, 500)

    cursor.execute(
        """
        INSERT INTO inventory (
            product_id,
            quantity
        )
        VALUES (%s, %s)
        """,
        (
            product_id,
            quantity
        )
    )

conn.commit()

print("Inventory data inserted successfully!")

## Data Quality

In [ ]:
# Customers

cursor.execute("""
    SELECT COUNT(*)
    FROM customers
""")

print("Customers:", cursor.fetchone()[0])

In [ ]:
# Products

cursor.execute("""
    SELECT COUNT(*)
    FROM products
""")

print("Products:", cursor.fetchone()[0])

In [ ]:
# Order

cursor.execute("""
    SELECT COUNT(*)
    FROM orders
""")

print("Orders:", cursor.fetchone()[0])

In [ ]:
# Order Item

cursor.execute("""
    SELECT COUNT(*)
    FROM order_items
""")

print("Order Items:", cursor.fetchone()[0])

In [ ]:
# Payments

cursor.execute("""
    SELECT COUNT(*)
    FROM payments
""")

print("Payments:", cursor.fetchone()[0])

In [ ]:
# Inventory

cursor.execute("""
    SELECT COUNT(*)
    FROM inventory
""")

print("Inventory:", cursor.fetchone()[0])

## Check FKs

### Product -> Categories

In [ ]:
cursor.execute("""
    SELECT COUNT(*)
    FROM products p
    LEFT JOIN categories c
        ON p.category_id = c.category_id
    WHERE c.category_id IS NULL
""")

print(
    "Products without category:",
    cursor.fetchone()[0]
)

### Orders → Customers

In [ ]:
cursor.execute("""
    SELECT COUNT(*)
    FROM orders o
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""")

print(
    "Orders without customer:",
    cursor.fetchone()[0]
)

### Order Items → Orders

In [ ]:
cursor.execute("""
    SELECT COUNT(*)
    FROM order_items oi
    LEFT JOIN orders o
        ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
""")

print(
    "Order items without order:",
    cursor.fetchone()[0]
)

### Order Items → Products

In [ ]:
cursor.execute("""
    SELECT COUNT(*)
    FROM order_items oi
    LEFT JOIN products p
        ON oi.product_id = p.product_id
    WHERE p.product_id IS NULL
""")

print(
    "Order items without product:",
    cursor.fetchone()[0]
)

## Close Conection

In [ ]:
cursor.close()
conn.close()

print("PostgreSQL connection closed.")